In [1]:
!pip install textblob pysentimiento langdetect -q
!python -m textblob.download_corpora 2>/dev/null
print("✓ Listo")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 10.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 36.5 MB/s eta 0:00:00
Finished.
✓ Listo


In [2]:
import pandas as pd
import numpy as np
from textblob import TextBlob
from pysentimiento import create_analyzer
import warnings
warnings.filterwarnings('ignore')

# Subir el CSV procesado
from google.colab import files
uploaded = files.upload()
import io
nombre = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[nombre]))

print(f"✓ Dataset cargado: {df.shape}")
print(f"  Español: {(df['idioma']=='es').sum()} reseñas")
print(f"  Inglés:  {(df['idioma']=='en').sum()} reseñas")

Saving tripadvisor_ecuador_procesado.csv to tripadvisor_ecuador_procesado.csv
✓ Dataset cargado: (43, 15)
  Español: 24 reseñas
  Inglés:  19 reseñas


In [3]:
# TextBlob para inglés — basado en léxico, sin GPU
# pysentimiento para español — modelo BERT entrenado en Twitter/reseñas

print("Cargando modelos de análisis de sentimiento...")
print("  TextBlob (EN): cargando...")
# TextBlob no requiere inicialización explícita

print("  pysentimiento (ES): cargando... (puede tardar 1-2 min)")
analizador_es = create_analyzer(task="sentiment", lang="es")

print("\n✓ Ambos modelos listos")
print("""
Estrategia de análisis:
  - Inglés  → TextBlob: polarity [-1, 1] + subjectivity [0, 1]
  - Español → pysentimiento: POS / NEG / NEU con probabilidades
""")

Cargando modelos de análisis de sentimiento...
  TextBlob (EN): cargando...
  pysentimiento (ES): cargando... (puede tardar 1-2 min)


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]


✓ Ambos modelos listos

Estrategia de análisis:
  - Inglés  → TextBlob: polarity [-1, 1] + subjectivity [0, 1]
  - Español → pysentimiento: POS / NEG / NEU con probabilidades



In [4]:
def analizar_ingles(texto):
    """
    TextBlob para reseñas en inglés.
    Retorna polarity [-1,1], subjectivity [0,1]
    y etiqueta de sentimiento.
    """
    if not isinstance(texto, str) or texto.strip() == '':
        return {'polaridad': 0.0, 'subjetividad': 0.5,
                'sentimiento': 'NEU', 'confianza': 0.0}
    blob = TextBlob(texto)
    pol  = blob.sentiment.polarity
    subj = blob.sentiment.subjectivity

    if pol > 0.1:
        etiqueta = 'POS'
    elif pol < -0.1:
        etiqueta = 'NEG'
    else:
        etiqueta = 'NEU'

    # Confianza aproximada: distancia desde 0
    confianza = min(abs(pol) * 2, 1.0)

    return {
        'polaridad':    round(pol, 4),
        'subjetividad': round(subj, 4),
        'sentimiento':  etiqueta,
        'confianza':    round(confianza, 4),
    }


def analizar_espanol(texto, analizador):
    """
    pysentimiento para reseñas en español.
    Retorna etiqueta (POS/NEG/NEU) y probabilidades.
    """
    if not isinstance(texto, str) or texto.strip() == '':
        return {'polaridad': 0.0, 'subjetividad': None,
                'sentimiento': 'NEU', 'confianza': 0.0}
    try:
        # pysentimiento trunca automáticamente a 512 tokens
        resultado = analizador.predict(texto[:512])
        etiqueta  = resultado.output  # 'POS', 'NEG', 'NEU'
        probs     = resultado.probas  # dict con probabilidades

        # Convertir a polaridad numérica para comparabilidad
        pol = probs.get('POS', 0) - probs.get('NEG', 0)
        confianza = max(probs.values())

        return {
            'polaridad':    round(pol, 4),
            'subjetividad': None,  # pysentimiento no calcula esto
            'sentimiento':  etiqueta,
            'confianza':    round(confianza, 4),
        }
    except Exception as e:
        return {'polaridad': 0.0, 'subjetividad': None,
                'sentimiento': 'NEU', 'confianza': 0.0}


print("✓ Funciones definidas")

✓ Funciones definidas


In [5]:
print("Analizando sentimiento...\n")

polaridades   = []
subjetividades = []
sentimientos  = []
confianzas    = []

for _, fila in df.iterrows():
    texto  = fila['texto_completo']
    idioma = fila['idioma']

    if idioma == 'en':
        res = analizar_ingles(texto)
    else:
        res = analizar_espanol(texto, analizador_es)

    polaridades.append(res['polaridad'])
    subjetividades.append(res['subjetividad'])
    sentimientos.append(res['sentimiento'])
    confianzas.append(res['confianza'])

df['polaridad_nlp']    = polaridades
df['subjetividad_nlp'] = subjetividades
df['sentimiento_nlp']  = sentimientos
df['confianza_nlp']    = confianzas

print("✓ Análisis completado")
print(f"\nDistribución de sentimiento (NLP):")
dist = df['sentimiento_nlp'].value_counts()
for sent, n in dist.items():
    print(f"  {sent}: {n} ({n/len(df)*100:.1f}%)")

Analizando sentimiento...

✓ Análisis completado

Distribución de sentimiento (NLP):
  POS: 34 (79.1%)
  NEG: 7 (16.3%)
  NEU: 2 (4.7%)


In [6]:
# El rating (1-5 estrellas) es nuestro ground truth
# Comparamos si el modelo NLP coincide con el rating

def rating_a_sentimiento(r):
    if pd.isna(r): return None
    if r >= 4:     return 'POS'
    if r == 3:     return 'NEU'
    return 'NEG'

df['sentimiento_rating'] = df['rating'].apply(rating_a_sentimiento)

# Solo comparar donde tenemos ambos
df_val = df[df['sentimiento_rating'].notna()].copy()

coincidencias = (df_val['sentimiento_nlp'] == df_val['sentimiento_rating']).sum()
total_val     = len(df_val)
precision     = coincidencias / total_val * 100

print("=" * 55)
print("VALIDACIÓN DEL MODELO NLP vs RATING (ground truth)")
print("=" * 55)
print(f"\nTotal reseñas con rating: {total_val}")
print(f"Coincidencias:            {coincidencias}")
print(f"Precisión global:         {precision:.1f}%")

print("\nDetalle de coincidencia por sentimiento:")
for sent in ['POS', 'NEU', 'NEG']:
    subset = df_val[df_val['sentimiento_rating'] == sent]
    if len(subset) == 0:
        continue
    aciertos = (subset['sentimiento_nlp'] == sent).sum()
    print(f"  {sent}: {aciertos}/{len(subset)} "
          f"({aciertos/len(subset)*100:.0f}%)")

print("\nMatriz de confusión (NLP vs Rating):")
confusion = pd.crosstab(
    df_val['sentimiento_rating'],
    df_val['sentimiento_nlp'],
    rownames=['Rating real'],
    colnames=['NLP predicho']
)
print(confusion.to_string())

VALIDACIÓN DEL MODELO NLP vs RATING (ground truth)

Total reseñas con rating: 43
Coincidencias:            32
Precisión global:         74.4%

Detalle de coincidencia por sentimiento:
  POS: 27/28 (96%)
  NEU: 0/9 (0%)
  NEG: 5/6 (83%)

Matriz de confusión (NLP vs Rating):
NLP predicho  NEG  NEU  POS
Rating real                
NEG             5    1    0
NEU             2    0    7
POS             0    1   27


In [7]:
print("=" * 55)
print("SENTIMIENTO POR DESTINO")
print("=" * 55)

resumen_destino = df.groupby('destino').agg(
    n_resenas       = ('sentimiento_nlp', 'count'),
    rating_promedio = ('rating', 'mean'),
    polaridad_media = ('polaridad_nlp', 'mean'),
    pct_positivo    = ('sentimiento_nlp', lambda x: (x=='POS').mean()*100),
    pct_neutro      = ('sentimiento_nlp', lambda x: (x=='NEU').mean()*100),
    pct_negativo    = ('sentimiento_nlp', lambda x: (x=='NEG').mean()*100),
).round(2).sort_values('polaridad_media', ascending=False)

print(resumen_destino.to_string())

print("\n" + "=" * 55)
print("SENTIMIENTO POR IDIOMA")
print("=" * 55)

resumen_idioma = df.groupby('idioma').agg(
    n_resenas       = ('sentimiento_nlp', 'count'),
    polaridad_media = ('polaridad_nlp', 'mean'),
    pct_positivo    = ('sentimiento_nlp', lambda x: (x=='POS').mean()*100),
    pct_negativo    = ('sentimiento_nlp', lambda x: (x=='NEG').mean()*100),
).round(2)

print(resumen_idioma.to_string())

SENTIMIENTO POR DESTINO
           n_resenas  rating_promedio  polaridad_media  pct_positivo  pct_neutro  pct_negativo
destino                                                                                       
Mindo              4             4.25             0.70         100.0         0.0           0.0
Cuenca             6             4.17             0.62         100.0         0.0           0.0
Quito              8             3.75             0.46          75.0        12.5          12.5
Otavalo            4             4.25             0.38          75.0         0.0          25.0
Banos              5             4.00             0.31          80.0         0.0          20.0
Montanita          4             3.25             0.20          75.0         0.0          25.0
Guayaquil          4             3.25             0.16          75.0         0.0          25.0
Galapagos          8             4.00             0.11          62.5        12.5          25.0

SENTIMIENTO POR IDIOMA
  

In [8]:
nombre_sentimiento = 'tripadvisor_ecuador_sentimiento.csv'
df.to_csv(nombre_sentimiento, index=False, encoding='utf-8-sig')

print(f"✓ Dataset con sentimiento guardado: {nombre_sentimiento}")
print(f"\nColumnas del dataset final:")
for col in df.columns:
    print(f"  - {col}")

from google.colab import files
files.download(nombre_sentimiento)
print("\n✓ Descargado — guárdalo en data/processed/")

✓ Dataset con sentimiento guardado: tripadvisor_ecuador_sentimiento.csv

Columnas del dataset final:
  - destino
  - categoria
  - region
  - titulo
  - texto
  - rating
  - fecha_raw
  - fecha_extraccion
  - titulo_limpio
  - texto_limpio
  - texto_completo
  - longitud_texto
  - idioma
  - sentimiento_rating
  - anio
  - polaridad_nlp
  - subjetividad_nlp
  - sentimiento_nlp
  - confianza_nlp


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Descargado — guárdalo en data/processed/
